# 10. Operating a bench

The same description of a bench is simulated, then operated, with the digital twin running beside the
equipment. The equipment here is the fictitious **LumosLabs** range: its emulators speak the LumosLabs protocol
over TCP, so LUMOS drives them exactly as it would drive real devices (same drivers, errors and latencies).

1. **UC3**: describe a bench (laser, shutter, lens on a stage, camera), record a sequence in simulation, then
   replay it on the equipment and compare the measured and simulated images;
2. safety: limits, emergency stop;
3. **UC4**: calibrate the phase of a photonic chip in operation, against the twin.

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import math
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from lumos import Context
from lumos.core.driver import DriverStopped, LimitError
from lumos.elements.detectors import Camera, PowerMeter
from lumos.elements.fibers import Fiber
from lumos.elements.lab import LaserSource, LinearStage, PhaseController, Shutter
from lumos.elements.optics import FreeSpace, ThinLens
from lumos.elements.photonics import PhaseShifter, YJunction, bracewell
from lumos.fields import Grid
from lumos.hardware.discovery import associate, discover
from lumos.hardware.emulators import EmulatorServer, create_emulator
from lumos.processing import calibration

## The equipment

In a laboratory, the devices are plugged into the computer. Here we start their emulators in this process
(`lumos emulators start` does the same from a terminal) and **discover** them: LUMOS asks every address
`*IDN?` and keeps the devices that answer.

In [ ]:
models = ["LL-LD", "LL-SH1", "LL-LS100", "LL-CAM", "LL-PCU", "LL-PM"]
servers = [
    EmulatorServer(create_emulator(m, serial=f"SN{i:04d}", latency=0.002)).start()
    for i, m in enumerate(models, start=1)
]
devices = discover(tcp=[server.url for server in servers])
for device in devices:
    print(f"{device.model:9s} {device.serial}  {device.url}")

## UC3: one bench, simulated then operated

A laser goes through a shutter and a lens carried by a stage, which focuses it on a camera. `ctx.link` makes
the propagation distance follow the stage, so that the twin sees the same focus as the bench. The camera is
described with the nominal parameters of the `LL-CAM` (gain, bias, dark current, read noise).

In [ ]:
def bench(mode):
    ctx = Context(mode=mode, name="focusing bench")
    laser = ctx.add(
        LaserSource(name="laser", power=1e-9 * u.mW, waist=1 * u.mm, n_pixels=128, pixel_scale=40 * u.um)
    )
    shutter = ctx.add(Shutter(name="shutter"))
    lens = ctx.add(ThinLens(name="lens", focal_length=200 * u.mm))
    gap = ctx.add(FreeSpace(name="gap", distance=180 * u.mm, output_grid=Grid(32, dx=8 * u.um)))
    camera = ctx.add(
        Camera(
            name="camera",
            exposure=0.02 * u.s,
            gain=2 * u.electron / u.adu,
            bias=100 * u.adu,
            dark_current=5 * u.electron / u.s,
            read_noise=3 * u.electron,
            quantum_efficiency=[1.0],
            noise=False,
        )
    )
    laser >> shutter >> lens >> gap >> camera
    ctx.add(LinearStage(name="stage", velocity=50 * u.mm / u.s, acceleration=500 * u.mm / u.s**2))
    ctx.link("stage.position", "gap.distance", offset=180 * u.mm)
    return ctx


def procedure(ctx):
    ctx["laser"].emit(True)
    ctx["shutter"].open()
    ctx["stage"].home()
    ctx["stage"].move(20 * u.mm)  # the lens is 200 mm from the camera: focus
    ctx["camera"].expose()


simulation = bench("simulation")
with simulation.record() as sequence:
    procedure(simulation)
simulated = simulation.run(["camera.out"])["camera.out"]
print(sequence, "lasting", simulation.clock.elapsed)
[str(step.command) for step in sequence]

The same sequence is replayed on the equipment. Switching the laser on is a dangerous action: the driver
asks for a confirmation (here a function that accepts; in the web interface or through an AI assistant, a
human answers). `operation_guard` brings everything to a safe state if anything goes wrong.

In [ ]:
operation = bench("operation")
associate(operation, devices, assignments={})
operation["laser"].driver.confirm = lambda command: True
with operation.operation_guard():
    operation.replay(sequence)
    result = operation.run(["camera.out"], archive="bench-operation")

same = [e.command for e in operation.journal.entries("action")] == [s.command for s in sequence]
print("same commands as in simulation:", same)
for entry in operation.journal.entries("action"):
    print(f"{entry.command!s:28s} predicted {entry.predicted:7.4f} s, measured {entry.measured:7.4f} s")

In [ ]:
measured, twin, residual = result["camera.out"], result.twin["camera.out"], result.residuals["camera.out"]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, image, title in zip(
    axes, (measured, twin, residual), ("measured (LL-CAM)", "digital twin", "residual"), strict=True
):
    limit = np.abs(image.data.value).max() if title == "residual" else None
    shown = ax.imshow(
        image.data.value,
        origin="lower",
        cmap="RdBu_r" if limit else "inferno",
        vmin=-limit if limit else None,
        vmax=limit,
    )
    ax.set(title=f"{title} [adu]", xticks=[], yticks=[])
    fig.colorbar(shown, ax=ax, fraction=0.046)
fig.tight_layout()
result.archive.save_figure(
    fig, "measured_and_twin", data={"measured": measured.data.value, "twin": twin.data.value}
)
print(f"mean residual {residual.data.value.mean():.2f} adu (hot pixels and noise of the device)")

## Safety

Every command is checked against the limits of its driver, read from the device, even when it is sent directly
to the driver. An emergency stop interrupts a motion in progress, from any thread.

In [ ]:
stage = operation["stage"]
try:
    stage.move(150 * u.mm)
except LimitError as error:
    print("refused:", error)

stage.velocity = 10 * u.mm / u.s
interrupted = []


def slow_move():
    try:
        stage.move(60 * u.mm)
    except DriverStopped as error:
        interrupted.append(error)


stage_emulator = servers[models.index("LL-LS100")].emulator
start = stage_emulator.position
thread = threading.Thread(target=slow_move)
thread.start()
while abs(stage_emulator.position - start) < 2:  # let it travel 2 mm at 10 mm/s
    time.sleep(0.01)
operation.stop_all()
time.sleep(0.05)  # the shutter takes 10 ms to close
thread.join()
print(interrupted[0], "| shutter after the stop:", operation["shutter"].driver.query("state"))
for driver in operation.drivers().values():
    driver.reset()

## UC4: calibrating the phase of a chip

A laser injected in a fibre is split into the two arms of a Bracewell combiner. One arm has a fabrication
error of 0.4 rad; a heater driven by the `LL-PCU` controller corrects it (phase κV²). The dark output is read
by the `LL-PM` power meter. The calibration scans the heater phase and keeps the deepest null, **measured on
the equipment**; the twin shows what the model predicts at the same settings.

In [ ]:
chip = Context(mode="operation", name="chip calibration")
laser = chip.add(
    LaserSource(name="laser", power=1 * u.mW, waist=2.5 * u.um, pixel_scale=0.5 * u.um, n_pixels=64)
)
fiber = chip.add(Fiber(name="fiber", mode_model="gaussian"))
split = chip.add(YJunction(name="split", mode="split", channels=[0]))
error = chip.add(PhaseShifter(name="fabrication", phase=0.4 * u.rad, achromatic=True, channels=[1]))
heater = chip.add(PhaseShifter(name="heater", phase=0 * u.rad, achromatic=True, channels=[1]))
laser >> fiber
chip.connect(fiber.port("modes"), split.port("in"))
split >> error >> heater >> bracewell(name="combiner") >> chip.add(PowerMeter(name="meter", channel=0))
pcu = chip.add(PhaseController(name="pcu"))
chip.link("pcu.voltages", "heater.phase", function=lambda voltages: pcu.phase(1))
chip.simulate(laser)  # the laser of this bench has no driver
laser.emit(True)
associate(chip, devices)
kappa = math.pi / 25 * u.rad / u.V**2


def null(phase):
    pcu.set_voltage(1, np.sqrt(max(phase[0], 0) * u.rad / kappa))
    return float(chip.run(["meter.out"])["meter.out"].data.to_value(u.W)[0])


fit = calibration.fit_phases(null, initial=[2 * math.pi] * u.rad, method="scan", points=21, iterations=5)
print(
    f"heater phase {fit.phases[0]:.4f} (expected {2 * math.pi - 0.4:.4f} rad), "
    f"voltage {np.sqrt(fit.phases[0] / kappa):.3f}, {fit.evaluations} measurements"
)

In [ ]:
phases = np.linspace(4.5, 7.3, 15)
real, model = [], []
for phase in phases:
    pcu.set_voltage(1, np.sqrt(phase * u.rad / kappa))
    run = chip.run(["meter.out"])
    real.append(run["meter.out"].data.to_value(u.W)[0])
    model.append(run.twin["meter.out"].data.to_value(u.W)[0])
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(phases, np.array(real) * 1e6, "o", label="measured (LL-PM)")
ax.plot(phases, np.array(model) * 1e6, "-", label="digital twin")
ax.axvline(fit.phases[0].value, color="gray", ls=":", label="calibrated")
ax.set(xlabel="heater phase [rad]", ylabel="dark output [µW]", yscale="log")
ax.legend()
fig.tight_layout()
result.archive.save_figure(fig, "chip_calibration", data={"phase": phases, "measured": real, "twin": model})

The measured null follows the twin, up to the gain error (2 %), the offset (1 nW) and the noise of the
power meter: the controlled gap between the bench and its model.

In [ ]:
for context in (operation, chip):
    for driver in context.drivers().values():
        driver.disconnect()
for server in servers:
    server.stop()